# Fire analysis


## Fire risk

Using [Open Climate Risk data](https://source.coop/carbonplan/carbonplan-ocr). Retrieval adapted from [their documentation](https://docs.carbonplan.org/ocr/en/latest/how-to/work-with-data.html#raster-xarray).


In [1]:
import icechunk
import xarray as xr

# Configure S3 storage for the Icechunk repository
version = "v1.1.0"
storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix=f"carbonplan/carbonplan-ocr/output/fire-risk/tensor/production/{version}/ocr.icechunk",
    region="us-west-2",
    anonymous=True,
)

# Open the repository
repo = icechunk.Repository.open(storage)

# Create a read-only session on the main branch
session = repo.readonly_session("main")

### Open the dataset


In [2]:
ds = xr.open_dataset(session.store, engine="zarr", chunks={})
ds

<xarray.Dataset> Size: 652GB
Dimensions:        (latitude: 97579, longitude: 208881)
Coordinates:
  * latitude       (latitude) float64 781kB 22.43 22.43 22.43 ... 52.48 52.48
  * longitude      (longitude) float64 2MB -128.4 -128.4 ... -64.05 -64.05
Data variables:
    bp_2011        (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    bp_2011_riley  (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    bp_2047        (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    rps_2011       (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    rps_2047       (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    bp_2047_riley  (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    crps_scott     (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    rps_scott      (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
Attributes:
    version:          1.1.0
    provider:         CarbonPlan
    terms_of_access:  https://docs.carbonplan.org/ocr/en/latest/terms-of-data...
    data_sources:     https://docs.carbonplan.org/ocr/en/latest/reference/dat...
    license_name:     CC-BY-4.0
    license_url:      https://creativecommons.org/licenses/by/4.0/

### Select a spatial subset

Extract data for a specific geographic region using coordinate slicing:


In [3]:
# Example: Select data for California region
california_subset = ds.sel(
    latitude=slice(32, 42),  # Southern to Northern California
    longitude=slice(-125, -114),  # Western to Eastern California
)

california_subset

<xarray.Dataset> Size: 37GB
Dimensions:        (latitude: 32468, longitude: 35715)
Coordinates:
  * latitude       (latitude) float64 260kB 32.0 32.0 32.0 ... 42.0 42.0 42.0
  * longitude      (longitude) float64 286kB -125.0 -125.0 ... -114.0 -114.0
Data variables:
    bp_2011        (latitude, longitude) float32 5GB dask.array<chunksize=(4922, 2501), meta=np.ndarray>
    bp_2011_riley  (latitude, longitude) float32 5GB dask.array<chunksize=(4922, 2501), meta=np.ndarray>
    bp_2047        (latitude, longitude) float32 5GB dask.array<chunksize=(4922, 2501), meta=np.ndarray>
    rps_2011       (latitude, longitude) float32 5GB dask.array<chunksize=(4922, 2501), meta=np.ndarray>
    rps_2047       (latitude, longitude) float32 5GB dask.array<chunksize=(4922, 2501), meta=np.ndarray>
    bp_2047_riley  (latitude, longitude) float32 5GB dask.array<chunksize=(4922, 2501), meta=np.ndarray>
    crps_scott     (latitude, longitude) float32 5GB dask.array<chunksize=(4922, 2501), meta=np.ndarray>
    rps_scott      (latitude, longitude) float32 5GB dask.array<chunksize=(4922, 2501), meta=np.ndarray>
Attributes:
    version:          1.1.0
    provider:         CarbonPlan
    terms_of_access:  https://docs.carbonplan.org/ocr/en/latest/terms-of-data...
    data_sources:     https://docs.carbonplan.org/ocr/en/latest/reference/dat...
    license_name:     CC-BY-4.0
    license_url:      https://creativecommons.org/licenses/by/4.0/

In [4]:
california_subset["bp_2011"][:100].to_dataframe()


bp_2011
latitude  longitude           
32.000102 -124.999778      NaN
          -124.999470      NaN
          -124.999162      NaN
          -124.998854      NaN
          -124.998546      NaN
...                        ...
32.030593 -114.001316      NaN
          -114.001008      NaN
          -114.000700      NaN
          -114.000392      NaN
          -114.000084      NaN

[3571500 rows x 1 columns]

## Active fires


### SQL Magic setup

Using [JupySQL](https://jupysql.readthedocs.io/). [Docs from DuckDB.](https://duckdb.org/docs/current/guides/python/jupyter)


In [9]:
%config SqlMagic.displaycon = False

In [6]:
import duckdb

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

Tip: You may define configurations in /workspaces/ptps-wildfire-demo/pyproject.toml or /home/codespace/.jupysql/config.

Did not find user configurations in /workspaces/ptps-wildfire-demo/pyproject.toml.

### Query data

> Each MODIS active fire/thermal hotspot location represents the center of a 1km pixel that is flagged by the algorithm as containing one or more fires within the pixel.

https://www.earthdata.nasa.gov/data/tools/firms

https://firms.modaps.eosdis.nasa.gov/active_fire/#firms-txt


In [10]:
%%sql
SELECT *
FROM 'https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/csv/MODIS_C6_1_USA_contiguous_and_Hawaii_24h.csv'
LIMIT 10;


latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,confidence,version,bright_t31,frp,daynight
43.12318,-122.56252,315.25,4.68,1.98,2026-08-06,0009,A,54,6.1NRT,304.61,55.9,D
43.12629,-122.5703,317.44,4.68,1.98,2026-08-06,0009,A,67,6.1NRT,304.7,74.63,D
20.7642,-75.95139,315.98,1.04,1.02,2026-08-06,0139,T,91,6.1NRT,293.62,14.71,N
20.76293,-75.96105,305.64,1.04,1.02,2026-08-06,0139,T,34,6.1NRT,290.51,6.4,N
21.52231,-77.89965,306.77,1.01,1.0,2026-08-06,0139,T,60,6.1NRT,295.08,4.34,N
21.52098,-77.90908,305.44,1.01,1.0,2026-08-06,0139,T,52,6.1NRT,295.2,3.53,N
21.53267,-77.90722,312.15,1.01,1.0,2026-08-06,0139,T,82,6.1NRT,295.48,8.08,N
21.85873,-79.15553,303.39,1.08,1.04,2026-08-06,0139,T,55,6.1NRT,285.91,5.29,N
22.07219,-78.32543,310.6,1.02,1.01,2026-08-06,0139,T,80,6.1NRT,295.4,6.77,N
32.44772,-82.49261,304.39,1.13,1.06,2026-08-06,0143,T,56,6.1NRT,284.98,6.87,N
